# Setup

In [ ]:
import networkx as nx

In [ ]:
import os
import gzip
import pickle

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

# Process Panaroo Results to Generate Representative Fasta File

In [ ]:
PANAROO_RESULTS = '../../data/processed/panaroo_output/'

G = nx.read_gml(os.path.join(PANAROO_RESULTS, "final_graph.gml"))

In [ ]:
df_genes = pd.read_csv('../../data/processed/panaroo_output/gene_presence_absence.Rtab', sep='\t', index_col='Gene')

In [ ]:
print("Writing Representative Sequences to File for each Gene...")
with open(os.path.join(PANAROO_RESULTS, 'representative_alleles.faa'), 'w') as f:
    for node, attributes in tqdm(G.nodes.data(True)):
        name = attributes['name']

        prot_seqs = attributes['protein'].split(';')
        max_len = 0
        max_prot = ""

        for prot in prot_seqs:
            if "*" in prot:
                continue

            if len(prot) > max_len:
                max_len = len(prot)
                max_prot = prot

        if max_prot == "":
            print(prot_seqs)
            break
        prot_seq = max_prot

        f.write(">" + name + '\n')
        f.write(prot_seq+'\n')

# Execute new eggNOG annotation

__The following are run in a linux terminal session:__

`tmux new -s 'eggNOG-annot'`

`conda activate emapper` [installation instructions](https://github.com/eggnogdb/eggnog-mapper)

`python emapper.py -o Ebacter --tax_scope Gammaproteobacteria --tax_scope_mode Bacteria -i /mnt/craig/pan_phylon/Enterobacter/zenodo_data_mSpectrum_revision/data/processed/panaroo_output/representative_alleles.faa --output_dir /mnt/craig/pan_phylon/Enterobacter/zenodo_data_mSpectrum_revision/data/processed/eggNOG --cpu 20`

# Postprocess resultant file

In [ ]:
# Read in file, skipping the first 4 rows (not needed)
df_eggnog = pd.read_csv(
    '../../data/processed/eggNOG/Ebacter.emapper.annotations',
    sep='\t',
    skiprows=4
)

# Remove the last 3 rows (not needed)
df_eggnog = df_eggnog[:-3]

# Rename "#query" to "allele"
df_eggnog.rename(columns={'#query': 'allele'}, inplace=True)

# Add in gene column
df_eggnog['gene'] = df_eggnog.allele.apply(lambda x: x)

# Set gene as the index
df_eggnog.set_index('gene', inplace=True)

print(f'initial shape: {df_eggnog.shape}')

df_eggnog

In [ ]:
len(set(df_genes.index) - set(df_eggnog.index)) # genes which were dropped by eggNOG (no hits)

In [ ]:
assert len(set(df_eggnog.index) - set(df_genes.index)) == 0 # genes in eggNOG which aren't in the pangenome (should be zero)

In [ ]:
# Add in dropped genes (genes which eggNOG drops because no OG could be found)
df_drop = pd.DataFrame(index=sorted(set(df_genes.index) - set(df_eggnog.index)), columns=df_eggnog.columns)
df_drop.index.name = 'gene'
df_drop.fillna('-', inplace=True)
df_eggnog = pd.concat([df_eggnog, df_drop])
print(f'final shape: {df_eggnog.shape}')

In [ ]:
df_eggnog

# Save eggNOG annotations

In [ ]:
df_eggnog.to_csv('../../data/processed/df_eggnog.csv')

# Generate Header to Allele Translations

In [ ]:
panaroo_results_path = '../../data/processed/panaroo_output'

In [ ]:
panaroo_df_genes = pd.read_csv(os.path.join(panaroo_results_path, 'gene_presence_absence.csv'), index_col='Gene', low_memory = False, dtype=object).drop(['Non-unique Gene name', 'Annotation'], axis=1)
gene_information = pd.read_csv(os.path.join(panaroo_results_path, 'gene_data.csv'), dtype=object)
genomes = list(panaroo_df_genes.columns)
graph = G

In [ ]:
gene_information = gene_information.set_index('clustering_id')
panaroo_dict = {}  # use a plain dict first
header_dict = {}
for node, attributes in tqdm(G.nodes.data(True)):
    name = attributes['name']
    seq_ids = attributes['seqIDs']
    for seq_id in seq_ids:
        panaroo_dict[seq_id] = name
        header_dict[gene_information.loc[seq_id, 'annotation_id']] = name

# Convert to pandas Series at the end
panaroo_header_to_allele = pd.Series(panaroo_dict)
header_to_allele = pd.Series(header_dict)

panaroo_header_to_allele.to_pickle("../../data/processed/panaroo_output/panaroo_header_to_allele.pickle.gz", compression="gzip")
header_to_allele.to_pickle("../../data/processed/panaroo_output/header_to_allele.pickle.gz", compression="gzip")

gene_information = gene_information.reset_index()

In [ ]:
num_members = {}
for node, attributes in tqdm(graph.nodes.data(True)):
    num_members[attributes['name']] = len(attributes['seqIDs'])

num_members = pd.Series(num_members)

# Save Paralog Information

In [ ]:
from collections import defaultdict
centroids_to_nodes = defaultdict(list)

In [ ]:
for node, attributes in graph.nodes.data(True):
    if attributes['paralog']:
        for centroid in attributes['centroid'].split(';'):
            centroids_to_nodes[centroid].append(attributes['name'])

In [ ]:
paralogous_groups_by_gene = defaultdict(set)

for key, item in centroids_to_nodes.items():
    for gene in item:
        paralogous_groups_by_gene[gene] = paralogous_groups_by_gene[gene].union(set(item))

In [ ]:
all_paralogs = set()
for genes in paralogous_groups_by_gene.values():
    all_paralogs = all_paralogs.union(set(genes))

In [ ]:
parent = {}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    parent[find(x)] = find(y)
    
# use union find to create disjoint sets of paralogs
for genes in paralogous_groups_by_gene.values():
    for gene in genes:
        if gene not in parent:
            parent[gene] = gene

for genes in paralogous_groups_by_gene.values():
    genes = list(genes)
    for i in range(1, len(genes)):
        union(genes[0], genes[i])

# Collect groups
groups = defaultdict(set)
for elem in parent:
    groups[find(elem)].add(elem)

# Final unique groupings
unique_groups = list(groups.values())
print(len(unique_groups))

In [ ]:
import pickle

# make a dictionary for checking paralogs and save for future use
genes_by_paralog_group = {}

for group in unique_groups:
    for gene in group:
        genes_by_paralog_group[gene] = group

with open('../../data/processed/infer_affinities/gene_by_paralog_group.pickle', "wb") as f:
    pickle.dump(genes_by_paralog_group, f)